In [84]:
## Peek Through the data

import duckdb
from pathlib import Path
path = Path("sgjobdata_old.duckdb")
# 1. delete (if it exists)
path.unlink(missing_ok=True)

con = duckdb.connect('sgjobdata_old.duckdb')      # creates file if it doesn't exist

con.execute("""
    CREATE or replace TABLE sgjobdata AS
    SELECT * FROM read_csv_auto('../data/SGJobData.csv')
""")
con.execute("CHECKPOINT")

con.close()
size_mb = Path("sgjobdata_old.duckdb").stat().st_size / 1024**2
print(f"sgjobdata_old.duckdb   {size_mb:.2f} MB")

sgjobdata_old.duckdb   70.01 MB


In [68]:
size_mb = Path("sgjobdata_old.duckdb").stat().st_size / 1024**2
print(f"sgjobdata_old.duckdb   {size_mb:.2f} MB")

sgjobdata_old.duckdb   139.26 MB


## File Size Comparison
69M May 12 07:40 sgjobdata.duckdb
273M May 11 22:15 SGJobData.csv


## Observations
Your CSV was ~240 MB; the DuckDB file is ~69 MB. That's roughly 3.5× compression, and it's totally expected. Several things are happening simultaneously.
1. Columnar storage beats row-based text
CSV stores data row-by-row as text. Every value — even a number — is stored as its string representation, with commas and newlines.
1,Singapore,2026-05-12,150.75
2,Singapore,2026-05-12,200.30
3,Malaysia, 2026-05-12,99.50
The string "150.75" takes 6 bytes. The number 150.75 as a float64 takes 8 bytes — but a float32 takes 4, and many numeric columns actually need even less.
DuckDB stores columns separately, in their natural binary type. No commas, no newlines, no string conversion overhead.
2. Dictionary encoding for repeated values
This is usually the biggest win. Look at a column like country:
CSV stores "Singapore" literally — 9 bytes — once per row. For 1M rows: 9 MB just for that one column.
DuckDB sees that country has only ~10 unique values and switches to dictionary encoding: store the distinct values once, then store each row as a tiny integer pointer (1 or 2 bytes). The same column drops to ~1-2 MB.
Any column with low cardinality — status, tier, country, category, event_type — gets this treatment automatically. The pandas equivalent is converting to category dtype, but DuckDB does it for you transparently.
3. Run-length encoding (RLE) for sorted/repeated data
If a column has long runs of the same value (e.g. date columns where many rows share the same date), DuckDB encodes them as "value × count" instead of repeating. A column with 100,000 consecutive "2026-05-12" entries collapses to essentially one entry plus a counter.
4. Bit-packing for small integers
If a column holds ages 0-127, DuckDB doesn't waste 8 bytes per row (int64). It uses just enough bits — often 1 byte (int8) or even fewer bits packed together. For columns that fit in small ranges, this is 4-8× smaller.
5. Lightweight compression on top
After all the above, DuckDB applies block-level compression (its default is a mix of techniques including FSST for strings, plus optional Snappy/zstd). Stacks on top of the encoding wins.
6. No more text-parsing overhead
CSV pays for human readability:

Numbers as text: "1234567" = 7 bytes vs 1234567 as int32 = 4 bytes
Booleans as "true"/"false": 4-5 bytes vs 1 bit
Timestamps as strings: "2026-05-12 14:23:45" = 19 bytes vs int64 = 8 bytes
Quotes, escapes, delimiters: pure overhead

Strip all that out and you're already at half the size before any compression kicks in.
A typical breakdown for a 240 MB → 69 MB conversion

In [3]:
## Peek Memory Usage
import pandas as pd

# Just the first 1000 rows to understand structure
sample = pd.read_csv('../data/SGJobData.csv', nrows=10000000)

print(sample.shape)
print(sample.dtypes)
print(sample.head())
print(sample.memory_usage(deep=True).sum() / 1024**2, 'MB')   # for these 1000 rows

(1048585, 22)
categories                             object
employmentTypes                        object
metadata_expiryDate                    object
metadata_isPostedOnBehalf                bool
metadata_jobPostId                     object
metadata_newPostingDate                object
metadata_originalPostingDate           object
metadata_repostCount                    int64
metadata_totalNumberJobApplication      int64
metadata_totalNumberOfView              int64
minimumYearsExperience                  int64
numberOfVacancies                       int64
occupationId                          float64
positionLevels                         object
postedCompany_name                     object
salary_maximum                          int64
salary_minimum                          int64
salary_type                            object
status_id                               int64
status_jobStatus                       object
title                                  object
average_salary      

In [4]:
sample.describe(include='all')


,categories,employmentTypes,metadata_expiryDate,metadata_isPostedOnBehalf,metadata_jobPostId,metadata_newPostingDate,metadata_originalPostingDate,metadata_repostCount,metadata_totalNumberJobApplication,metadata_totalNumberOfView,...,occupationId,positionLevels,postedCompany_name,salary_maximum,salary_minimum,salary_type,status_id,status_jobStatus,title,average_salary
count,1044597,1044597,1044597,1048585,1044597,1044597,1044597,1.048585e+06,1.048585e+06,1.048585e+06,...,0.0,1044597,1044597,1.048585e+06,1.048585e+06,1044597,1048585.0,1044597,1044597,1.048585e+06
unique,21125,8,453,2,1044597,431,603,NaN,NaN,NaN,...,NaN,9,53151,NaN,NaN,1,NaN,3,377084,NaN
top,"[{""id"":21,""category"":""Information Technology""}]",Permanent,2023-07-28,False,MCF-2023-0252866,2023-06-09,2023-07-14,NaN,NaN,NaN,...,NaN,Executive,THE SUPREME HR ADVISORY PTE. LTD.,NaN,NaN,Monthly,NaN,Open,SUPERVISOR,NaN
freq,92869,458139,4487,986717,1,4508,4029,NaN,NaN,NaN,...,NaN,253701,61638,NaN,NaN,1044597,NaN,902614,8331,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.472327e-02,2.136571e+00,2.674536e+01,...,NaN,NaN,NaN,5.723578e+03,3.815312e+03,NaN,0.0,NaN,NaN,4.769445e+03
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.822675e-01,1.062612e+01,8.262001e+01,...,NaN,NaN,NaN,5.018387e+04,3.172182e+03,NaN,0.0,NaN,NaN,2.547809e+04
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00,0.000000e+00,0.000000e+00,...,NaN,NaN,NaN,0.000000e+00,0.000000e+00,NaN,0.0,NaN,NaN,0.000000e+00
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00,0.000000e+00,1.000000e+00,...,NaN,NaN,NaN,3.300000e+03,2.500000e+03,NaN,0.0,NaN,NaN,2.900000e+03
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00,0.000000e+00,4.000000e+00,...,NaN,NaN,NaN,4.500000e+03,3.000000e+03,NaN,0.0,NaN,NaN,3.800000e+03
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00,1.000000e+00,1.700000e+01,...,NaN,NaN,NaN,6.500000e+03,4.500000e+03,NaN,0.0,NaN,NaN,5.500000e+03


In [5]:
sample.info(memory_usage='deep')
print(sample.memory_usage(deep=True).sum() / 1024**2, 'MB')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048585 entries, 0 to 1048584
Data columns (total 22 columns):
 #   Column                              Non-Null Count    Dtype  
---  ------                              --------------    -----  
 0   categories                          1044597 non-null  object 
 1   employmentTypes                     1044597 non-null  object 
 2   metadata_expiryDate                 1044597 non-null  object 
 3   metadata_isPostedOnBehalf           1048585 non-null  bool   
 4   metadata_jobPostId                  1044597 non-null  object 
 5   metadata_newPostingDate             1044597 non-null  object 
 6   metadata_originalPostingDate        1044597 non-null  object 
 7   metadata_repostCount                1048585 non-null  int64  
 8   metadata_totalNumberJobApplication  1048585 non-null  int64  
 9   metadata_totalNumberOfView          1048585 non-null  int64  
 10  minimumYearsExperience              1048585 non-null  int64  
 11  numberOfVac

In [6]:
sample.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048585 entries, 0 to 1048584
Data columns (total 22 columns):
 #   Column                              Non-Null Count    Dtype  
---  ------                              --------------    -----  
 0   categories                          1044597 non-null  object 
 1   employmentTypes                     1044597 non-null  object 
 2   metadata_expiryDate                 1044597 non-null  object 
 3   metadata_isPostedOnBehalf           1048585 non-null  bool   
 4   metadata_jobPostId                  1044597 non-null  object 
 5   metadata_newPostingDate             1044597 non-null  object 
 6   metadata_originalPostingDate        1044597 non-null  object 
 7   metadata_repostCount                1048585 non-null  int64  
 8   metadata_totalNumberJobApplication  1048585 non-null  int64  
 9   metadata_totalNumberOfView          1048585 non-null  int64  
 10  minimumYearsExperience              1048585 non-null  int64  
 11  numberOfVac

In [7]:
## Load from CSV , clean and save as DuckDB for future use
ori_csv_path = '../data/SGJobData.csv'
old_df = pd.read_csv(ori_csv_path)
print(old_df.memory_usage(deep=True).sum() / 1024**2, 'MB')

old_df.isna().sum() 
old_df.isna().all()  # Check if any value in each column is NaN

921.3089017868042 MB


categories                            False
employmentTypes                       False
metadata_expiryDate                   False
metadata_isPostedOnBehalf             False
metadata_jobPostId                    False
metadata_newPostingDate               False
metadata_originalPostingDate          False
metadata_repostCount                  False
metadata_totalNumberJobApplication    False
metadata_totalNumberOfView            False
minimumYearsExperience                False
numberOfVacancies                     False
occupationId                           True
positionLevels                        False
postedCompany_name                    False
salary_maximum                        False
salary_minimum                        False
salary_type                           False
status_id                             False
status_jobStatus                      False
title                                 False
average_salary                        False
dtype: bool

In [8]:

## status_id has 0 non-null values, so we can drop it
## occupation id is NA
## Salary type all are monthly, so we can drop it as well
dropped_df = old_df.drop(columns=['status_id', 'occupationId', 'salary_type'])



In [9]:
## Schema Definition
DTYPES = {
    'categories':                          'object',     # JSON string, parse later
    'employmentTypes':                     'category',
    'metadata_isPostedOnBehalf':           'bool',
    'metadata_jobPostId':                  'string',
    'metadata_repostCount':                'int8',
    'metadata_totalNumberJobApplication':  'int16',
    'metadata_totalNumberOfView':          'int16',
    'minimumYearsExperience':              'int8',
    'numberOfVacancies':                   'int16',
    'positionLevels':                      'category',
    'postedCompany_name':                  'category',
    'salary_maximum':                      'int32',
    'salary_minimum':                      'int32',
    'status_jobStatus':                    'category',
    'title':                               'string',
    'average_salary':                      'float32',
}
 
DATE_COLS = [
    'metadata_expiryDate',
    'metadata_newPostingDate',
    'metadata_originalPostingDate',
]
 
# Columns to drop entirely (constant / all-null / no info)
DROP_COLS = ['occupationId', 'salary_type', 'status_id']
 
# Read columns = everything in DTYPES + DATE_COLS, minus DROP_COLS
usecols = [c for c in list(DTYPES) + DATE_COLS if c not in DROP_COLS]


In [14]:
filtered_df = df = pd.read_csv(
    ori_csv_path,
    usecols=usecols,
    dtype=DTYPES,
    parse_dates=DATE_COLS,
)

print("Original DataFrame memory usage:", old_df.memory_usage(deep=True).sum() / 1024**2, 'MB')
print("Dropped DataFrame memory usage:", dropped_df.memory_usage(deep=True).sum() / 1024**2, 'MB') ## reduced appx 80 MB
print("Filtered DataFrame memory usage:", filtered_df.memory_usage(deep=True).sum() / 1024**2, 'MB') ## reduced appx 80 MB



Original DataFrame memory usage: 921.3089017868042 MB
Dropped DataFrame memory usage: 841.4299192428589 MB
Filtered DataFrame memory usage: 353.1520185470581 MB


In [11]:
clean_job_df = filtered_df.copy()

clean_job_df.to_parquet("../data/clean_job.parquet", compression="snappy")
clean_job_df.to_pickle("../data/clean_job.pkl")
clean_job_df.to_csv("../data/clean_job.csv", index=False) 

In [13]:
from pathlib import Path

for f in ["../data/clean_job.parquet", "../data/clean_job.pkl", "../data/clean_job.csv"]:
    size_mb = Path(f).stat().st_size / 1024**2
    print(f"{f:20s} {size_mb:8.2f} MB")


../data/clean_job.parquet    45.02 MB
../data/clean_job.pkl   119.25 MB
../data/clean_job.csv   261.70 MB


In [ ]:
files = {
    "parquet": "../data/clean_job.parquet",
    "pickle":  "../data/clean_job.pkl",
    "csv":     "../data/clean_job.csv",
}

loaders = {
    "parquet": pd.read_parquet,
    "pickle":  pd.read_pickle,
    "csv":     pd.read_csv,
}

for label, path in files.items():
    df = loaders[label](path)
    disk_mb = Path(path).stat().st_size / 1024**2
    mem_mb  = df.memory_usage(deep=True).sum() / 1024**2
    print(f"{label:8s}  disk: {disk_mb:7.2f} MB   in-memory: {mem_mb:7.2f} MB   rows: {len(df):,}")



parquet   disk:   45.02 MB   in-memory:  353.12 MB   rows: 1,048,585
pickle    disk:  119.25 MB   in-memory:  351.14 MB   rows: 1,048,585
csv       disk:  261.70 MB   in-memory:  841.43 MB   rows: 1,048,585


In [93]:
# ---- from PARQUET ----
# Option A: load via pandas, then hand it to duckdb
df_parquet = pd.read_parquet("../data/clean_job.parquet")

path_parquet_db = Path("sgjobdata_parquet.duckdb")
path_parquet_db.unlink(missing_ok=True)

con = duckdb.connect(str(path_parquet_db))
con.execute("CREATE OR REPLACE TABLE sgjobdata AS SELECT * FROM df_parquet")
con.execute("CHECKPOINT")
con.close()

In [ ]:
# ---- from CSV ----
# Option A: load via pandas, then hand it to duckdb
df_csv = pd.read_csv("../data/clean_job.csv")

path_csv_db = Path("sgjobdata_csv.duckdb")
path_csv_db.unlink(missing_ok=True)

con = duckdb.connect(str(path_csv_db))
con.execute("CREATE OR REPLACE TABLE sgjobdata AS SELECT * FROM df_csv")
con.execute("CHECKPOINT")
con.close()


In [89]:
# ---- from PKL ----
df_pkl = pd.read_pickle("../data/clean_job.pkl")

path_pkl_db = Path("sgjobdata_pkl.duckdb")
path_pkl_db.unlink(missing_ok=True)

con = duckdb.connect(str(path_pkl_db))
con.execute("CREATE OR REPLACE TABLE sgjobdata AS SELECT * FROM df_pkl")
con.execute("CHECKPOINT")
con.close()


## Memory Usage Comparison and File Size Comparison     

In [92]:
size_mb = Path("sgjobdata_pkl.duckdb").stat().st_size / 1024**2
print(f"sgjobdata_pkl.duckdb   {size_mb:.2f} MB")


size_mb = Path("sgjobdata_csv.duckdb").stat().st_size / 1024**2
print(f"sgjobdata_csv.duckdb  {size_mb:.2f} MB")


size_mb = Path("sgjobdata_parquet.duckdb").stat().st_size / 1024**2
print(f"sgjobdata_parquet.duckdb   {size_mb:.2f} MB")


sgjobdata_pkl.duckdb   66.76 MB
sgjobdata_csv.duckdb  71.51 MB
sgjobdata_parquet.duckdb   95.26 MB


In [ ]:
print("df_pkl:",df_pkl.memory_usage(deep=True).sum() / 1024**2, 'MB')
print("df_csv:",df_csv.memory_usage(deep=True).sum() / 1024**2, 'MB')
print("df_parquet:",df_parquet.memory_usage(deep=True).sum() / 1024**2, 'MB')

351.13568019866943 MB
841.4299192428589 MB
353.1215925216675 MB
